# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad5987/ML-/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)



## 1. Ranked Actions + Reason Codes

The action queue is intended as directional decision-support for content review. Pages are ranked using observable search-performance signals rather than treated as automatic recommendations.

The highest-priority pages are those with meaningful impressions but weak CTR, because these pages already receive search visibility while showing an opportunity for human review. Reason codes are used to explain why a page appears in the queue.

### Reason codes

- `HIGH_IMPRESSIONS_LOW_CTR` — the page receives substantial impressions but has a low measured CTR.
- `LOW_VISIBILITY` — the page has limited search impressions and may require further investigation before any action is recommended.
- `POSITION_OPPORTUNITY` — the page has measurable search visibility but its average position suggests there may be room for improvement.
- `REVIEW` — the available signals are not sufficient for a more specific recommendation.

The queue is ranked for review priority, not for automatic publishing or editing. A human should inspect the page context, search intent, existing content, and business relevance before taking action.

In [15]:
import duckdb

con = duckdb.connect()

In [16]:
con.sql(f"""
CREATE OR REPLACE TABLE daily AS
SELECT *
FROM read_parquet('{sample}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [17]:
con.sql("""
SELECT COUNT(*) AS total_rows
FROM daily
""").df()

,total_rows
0,11694072


In [18]:
# Build a practical, human-review action queue

queue = con.sql("""
SELECT
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,

    CASE
        WHEN gsc_impressions >= 1000
             AND (
                 100.0 * gsc_clicks /
                 NULLIF(gsc_impressions, 0)
             ) < 1
            THEN 'HIGH_IMPRESSIONS_LOW_CTR'

        WHEN gsc_impressions < 100
            THEN 'LOW_VISIBILITY'

        WHEN gsc_avg_position > 10
            THEN 'POSITION_OPPORTUNITY'

        ELSE 'REVIEW'
    END AS reason_code,

    CASE
        WHEN gsc_impressions >= 1000
             AND (
                 100.0 * gsc_clicks /
                 NULLIF(gsc_impressions, 0)
             ) < 1
            THEN 3

        WHEN gsc_impressions < 100
            THEN 1

        WHEN gsc_avg_position > 10
            THEN 2

        ELSE 1
    END AS priority_score

FROM daily
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
ORDER BY priority_score DESC, gsc_impressions DESC
LIMIT 100
""").df()

display(queue.head(20))

print("Queue rows:", len(queue))
print("\nReason-code distribution:")
display(queue["reason_code"].value_counts())

,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,reason_code,priority_score
0,2026-06-11,245826,890,6.326149,1772,685,HIGH_IMPRESSIONS_LOW_CTR,3
1,2026-06-29,49373,173,2.627995,147,96,HIGH_IMPRESSIONS_LOW_CTR,3
2,2026-06-30,48990,189,2.590467,159,99,HIGH_IMPRESSIONS_LOW_CTR,3
3,2026-06-26,48953,114,2.586277,84,52,HIGH_IMPRESSIONS_LOW_CTR,3
4,2026-06-12,46220,81,6.856274,312,153,HIGH_IMPRESSIONS_LOW_CTR,3
5,2026-06-30,45890,329,5.828089,822,413,HIGH_IMPRESSIONS_LOW_CTR,3
6,2026-06-25,42474,107,2.605735,112,57,HIGH_IMPRESSIONS_LOW_CTR,3
7,2026-06-27,41051,129,2.605150,83,60,HIGH_IMPRESSIONS_LOW_CTR,3
8,2026-06-29,35560,313,5.676069,743,377,HIGH_IMPRESSIONS_LOW_CTR,3
9,2026-06-24,30645,50,5.019710,92,43,HIGH_IMPRESSIONS_LOW_CTR,3


Queue rows: 100

Reason-code distribution:


,count
reason_code,
HIGH_IMPRESSIONS_LOW_CTR,100


## 2. Intended Use and Limits

### Intended use

This playbook is intended to help content teams prioritize pages for human review. It uses observed search-performance signals to identify pages that may deserve attention, especially pages with substantial search impressions and relatively low measured CTR.

The queue is a decision-support tool rather than an automated content optimization system. A reviewer can use the reason code and supporting metrics to decide which pages should be investigated first.

### Limits

The rankings should be treated as directional rather than definitive recommendations. The Week-5 model had a target-feature dependency because `gsc_impressions` and `gsc_clicks` contributed directly to the target definition. The validation also used a limited sample and date range.

The queue does not establish that changing a page will improve its performance. It also does not account for search intent, content quality, business priorities, seasonality, technical SEO issues, or changes in search behavior.

The output should therefore be used to prioritize investigation, not to automatically rewrite, publish, remove, or redirect content.

In [19]:
print("Queue rows available for decision-support:", len(queue))
print("Available queue columns:")
print(list(queue.columns))

Queue rows available for decision-support: 100
Available queue columns:
['report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'reason_code', 'priority_score']


## 3. Human Review + the No-Go List

### Human review rules

Every queued page must be reviewed by a person before any content action is taken. The reviewer should:

1. Check the page's search intent and whether the content matches the query intent.
2. Review the existing content for accuracy, usefulness, completeness, and quality.
3. Check whether the low CTR may be explained by the search result, SERP features, seasonality, or changes in search behavior.
4. Consider the page's business importance and whether updating it is worthwhile.
5. Verify that the proposed action does not remove useful information or create a worse experience for users.
6. Record the reason for the final decision rather than treating the queue as an automatic instruction.

### No-go list

The system should not automatically:

- Rewrite or publish content.
- Delete, redirect, or canonicalize pages.
- Change titles or metadata without human review.
- Make claims about ranking improvements or traffic increases.
- Override editorial, legal, brand, or business decisions.
- Treat a high-priority queue position as proof that a page needs to be changed.
- Make irreversible content changes based only on the model or queue.

The queue is for prioritization and decision-support. Final content decisions remain with a qualified human reviewer.

In [20]:
no_go_actions = [
    "automatic_rewrite",
    "automatic_publish",
    "automatic_delete",
    "automatic_redirect",
    "automatic_canonical_change"
]

print("No-go automated actions:")
for action in no_go_actions:
    print("-", action)

print("\nHuman review required:", True)

No-go automated actions:
- automatic_rewrite
- automatic_publish
- automatic_delete
- automatic_redirect
- automatic_canonical_change

Human review required: True


## 4. Monitoring / Retrain Triggers

The recommendations should be monitored regularly because search behavior, content performance, and the underlying data can change over time.

### Monitoring checks

- Track the distribution of impressions, clicks, CTR, and average position in the reviewed queue.
- Check whether the proportion of `HIGH_IMPRESSIONS_LOW_CTR` pages changes substantially over time.
- Review whether recommended pages continue to show the same observed patterns after human review.
- Monitor for missing, delayed, or unusual data in the source metrics.
- Compare new observations with the historical ranges used to create the queue.

### Retrain or review triggers

The model or ranking logic should be reconsidered when:

- Feature distributions change substantially from the data used previously.
- The meaning or availability of important features changes.
- The proportion of high-priority recommendations changes sharply.
- Human reviewers repeatedly disagree with the queue's prioritization.
- Search behavior or reporting definitions change.
- New evaluation data shows materially different performance.

Retraining should not happen automatically. A trigger should start a review of the data, target definition, leakage risks, and validation design before a new model is accepted.

In [21]:
print("Current queue size:", len(queue))

print("\nReason-code distribution:")
display(queue["reason_code"].value_counts())

print("\nMonitoring status:")
print("Review recommended before retraining:", True)

Current queue size: 100

Reason-code distribution:


,count
reason_code,
HIGH_IMPRESSIONS_LOW_CTR,100



Monitoring status:
Review recommended before retraining: True


## 5. Exports for the Paper

The ranked content-action queue is exported as a CSV so that it can be reused in the recommendations section of the research paper.

The exported queue contains the observed search-performance metrics, reason codes, and priority scores used to support human review. The file is generated directly by the notebook so the paper can trace the recommendations back to the analysis.

The queue is saved to:

`work/outputs/content_action_queue.csv`

The CSV is intended as an analysis output and decision-support artifact. It should not be treated as a production system output or as an automatic content-action list.

In [22]:
import pandas as pd
from pathlib import Path

# Create the required output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the ranked action queue
queue_path = output_dir / "content_action_queue.csv"
queue.to_csv(queue_path, index=False)

print(f"Exported queue to: {queue_path}")
print(f"Rows exported: {len(queue)}")

# Verify the exported file
check = pd.read_csv(queue_path)

print("\nExport verification:")
print(check.shape)
display(check.head(10))

Exported queue to: work/outputs/content_action_queue.csv
Rows exported: 100

Export verification:
(100, 8)


,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,reason_code,priority_score
0,2026-06-11,245826,890,6.326149,1772.0,685.0,HIGH_IMPRESSIONS_LOW_CTR,3
1,2026-06-29,49373,173,2.627995,147.0,96.0,HIGH_IMPRESSIONS_LOW_CTR,3
2,2026-06-30,48990,189,2.590467,159.0,99.0,HIGH_IMPRESSIONS_LOW_CTR,3
3,2026-06-26,48953,114,2.586277,84.0,52.0,HIGH_IMPRESSIONS_LOW_CTR,3
4,2026-06-12,46220,81,6.856274,312.0,153.0,HIGH_IMPRESSIONS_LOW_CTR,3
5,2026-06-30,45890,329,5.828089,822.0,413.0,HIGH_IMPRESSIONS_LOW_CTR,3
6,2026-06-25,42474,107,2.605735,112.0,57.0,HIGH_IMPRESSIONS_LOW_CTR,3
7,2026-06-27,41051,129,2.605150,83.0,60.0,HIGH_IMPRESSIONS_LOW_CTR,3
8,2026-06-29,35560,313,5.676069,743.0,377.0,HIGH_IMPRESSIONS_LOW_CTR,3
9,2026-06-24,30645,50,5.019710,92.0,43.0,HIGH_IMPRESSIONS_LOW_CTR,3


## Self-check


- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`